In [ ]:
!pip uninstall -y transformers peft accelerate tokenizers

!pip install -U transformers peft accelerate tokenizers

Found existing installation: tokenizers 0.23.2
Uninstalling tokenizers-0.23.2:
  Successfully uninstalled tokenizers-0.23.2
  Using cached transformers-5.17.0-py3-none-any.whl.metadata (32 kB)
  Using cached accelerate-1.15.0-py3-none-any.whl.metadata (19 kB)
  Using cached tokenizers-0.23.2-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.8 kB)
Using cached transformers-5.17.0-py3-none-any.whl (12.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 35.8 MB/s eta 0:00:00
Using cached accelerate-1.15.0-py3-none-any.whl (394 kB)
Using cached tokenizers-0.23.2-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.4 MB)


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPU is not enabled. Go to Runtime → Change runtime type → T4 GPU.")


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
!pip install -q -U transformers datasets accelerate scikit-learn seaborn matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 19.7 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Running on CPU")


PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [ ]:
from datasets import load_dataset

print("Loading Yelp dataset...")

yelp = load_dataset("fancyzhx/yelp_polarity")

print("Yelp loaded successfully!")
print(yelp)


Loading Yelp dataset...


Yelp loaded successfully!
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 560000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 38000
    })
})


In [ ]:
TRAIN_SIZE = 50000
YELP_TEST_SIZE = 10000

yelp_train = yelp["train"].shuffle(seed=42).select(
    range(TRAIN_SIZE)
)

yelp_test = yelp["test"].shuffle(seed=42).select(
    range(YELP_TEST_SIZE)
)

print("Yelp training samples:", len(yelp_train))
print("Yelp testing samples:", len(yelp_test))


Yelp training samples: 50000
Yelp testing samples: 10000


In [ ]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print("DistilBERT loaded successfully!")


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT loaded successfully!


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

print("Tokenizing Yelp training data...")

yelp_train_tokenized = yelp_train.map(
    tokenize_function,
    batched=True
)

print("Tokenizing Yelp test data...")

yelp_test_tokenized = yelp_test.map(
    tokenize_function,
    batched=True
)

print("Tokenization complete!")


Tokenizing Yelp training data...


Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Tokenizing Yelp test data...


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Tokenization complete!


In [ ]:
def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./yelp_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    report_to="none"
)

print("Training arguments created successfully!")



Training arguments created successfully!


In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss
1,0.195360,0.183186
2,0.135686,0.202008


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,0.195360,0.183186
2,0.135686,0.202008


TrainOutput(global_step=6250, training_loss=0.17869052337646485, metrics={'train_runtime': 1260.3964, 'train_samples_per_second': 79.34, 'train_steps_per_second': 4.959, 'total_flos': 3311684966400000.0, 'train_loss': 0.17869052337646485, 'epoch': 2.0})

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=yelp_train_tokenized,
    eval_dataset=yelp_test_tokenized
)

print("Trainer created successfully!")


Trainer created successfully!


In [ ]:
from datasets import load_dataset

print("Loading Amazon Reviews dataset...")

amazon = load_dataset("fancyzhx/amazon_polarity")

print("Amazon dataset loaded!")
print(amazon)

Loading Amazon Reviews dataset...
Amazon dataset loaded!
DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 3600000
    })
    test: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 400000
    })
})


In [ ]:
def combine_amazon_text(example):
    return {
        "text": example["title"] + " " + example["content"]
    }

amazon = amazon.map(combine_amazon_text)

print(amazon["test"][0])


{'label': 1, 'title': 'Great CD', 'content': 'My lovely Pat has one of the GREAT voices of her generation. I have listened to this CD for YEARS and I still LOVE IT. When I\'m in a good mood it makes me feel better. A bad mood just evaporates like sugar in the rain. This CD just oozes LIFE. Vocals are jusat STUUNNING and lyrics just kill. One of life\'s hidden gems. This is a desert isle CD in my book. Why she never made it big is just beyond me. Everytime I play this, no matter black, white, young, old, male, female EVERYBODY says one thing "Who was that singing ?"', 'text': 'Great CD My lovely Pat has one of the GREAT voices of her generation. I have listened to this CD for YEARS and I still LOVE IT. When I\'m in a good mood it makes me feel better. A bad mood just evaporates like sugar in the rain. This CD just oozes LIFE. Vocals are jusat STUUNNING and lyrics just kill. One of life\'s hidden gems. This is a desert isle CD in my book. Why she never made it big is just beyond me. Ever

In [ ]:
def tokenize_amazon(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=256
    )

In [ ]:
# Select 10,000 Amazon test reviews
amazon_test = amazon["test"].select(range(10000))

# Tokenize the reviews
amazon_test = amazon_test.map(
    tokenize_amazon,
    batched=True
)

# Change label name for Transformers
amazon_test = amazon_test.rename_column("label", "labels")

# Remove unnecessary text columns
amazon_test = amazon_test.remove_columns(
    ["title", "content", "text"]
)

print("Amazon test dataset is ready!")
print(amazon_test)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Amazon test dataset is ready!
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 10000
})


In [ ]:
print("trainer exists:", "trainer" in globals())
print("model exists:", "model" in globals())
print("tokenizer exists:", "tokenizer" in globals())

trainer exists: True
model exists: True
tokenizer exists: True


In [ ]:
# Re-tokenize Amazon with padding
def tokenize_amazon(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

amazon_test = amazon["test"].select(range(10000))
amazon_test = amazon_test.map(tokenize_amazon, batched=True)

amazon_test = amazon_test.rename_column("label", "labels")

amazon_test = amazon_test.remove_columns(
    ["title", "content", "text"]
)

print("Amazon test dataset fixed!")
print(amazon_test)


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Amazon test dataset fixed!
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 10000
})


In [ ]:
amazon_results = trainer.predict(amazon_test)

amazon_predictions = np.argmax(
    amazon_results.predictions,
    axis=1
)

amazon_labels = amazon_results.label_ids

print("Amazon prediction completed!")

Amazon prediction completed!


In [ ]:
accuracy = accuracy_score(
    amazon_labels,
    amazon_predictions
)

precision, recall, f1, _ = precision_recall_fscore_support(
    amazon_labels,
    amazon_predictions,
    average="binary"
)

print("\n===== AMAZON RESULTS =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")



===== AMAZON RESULTS =====
Accuracy : 0.9231
Precision: 0.9366
Recall   : 0.9116
F1 Score : 0.9240
